In [1]:
import pandas as pd
import ast
import os
import warnings

warnings.filterwarnings("ignore")

def extract_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [2]:
from Bio import SeqIO
from tqdm import tqdm

for genus_name in keep_genus:
    target_dir = rf"/active-data/analysis_results/chr_pla/genus/IMG_PR_plasmid/{genus_name}"
    contig_data = pd.read_csv(f'{target_dir}/IMGPR_plasmid_data.tsv', sep='\t')
    contig_data = contig_data[contig_data['host_taxonomy'].str.contains(f'g__{genus_name}', na=False)]
    Meta_data = contig_data[contig_data['source_type']=='Metagenome']
    contig_list = Meta_data['plasmid_id'].to_list()
    
    nucl_file = f"{target_dir}/IMGPR_nucl.fasta"
    meta_pla = open(f"{target_dir}/IMGPR_metagenome_plasmids.fasta", "w+")
    with tqdm(total = len(contig_data), desc=f'Metagenome plasmids ({genus_name})', leave=True, ncols=100, unit='B', unit_scale=True) as pbar:
        with open(nucl_file, 'r') as handle:
            seq_records = SeqIO.parse(handle, 'fasta')
            for record in seq_records:
                if record.id.split('|')[0] in contig_list:
                    SeqIO.write(record, meta_pla, "fasta")
                pbar.update(1)

Metagenome plasmids (Escherichia): 100%|███████████████████████| 24.4k/24.4k [00:03<00:00, 6.39kB/s]
Metagenome plasmids (Klebsiella): 100%|████████████████████████| 5.90k/5.90k [00:01<00:00, 3.33kB/s]
Metagenome plasmids (Staphylococcus): 100%|████████████████████| 12.9k/12.9k [00:02<00:00, 5.28kB/s]
Metagenome plasmids (Pseudomonas): 100%|███████████████████████| 4.98k/4.98k [00:01<00:00, 2.60kB/s]
Metagenome plasmids (Bacillus): 100%|██████████████████████████| 5.80k/5.80k [00:01<00:00, 4.12kB/s]
Metagenome plasmids (Salmonella): 100%|████████████████████████| 4.58k/4.58k [00:00<00:00, 5.00kB/s]
Metagenome plasmids (Streptococcus): 100%|█████████████████████| 4.74k/4.74k [00:01<00:00, 2.56kB/s]
Metagenome plasmids (Streptomyces): 100%|██████████████████████| 2.75k/2.75k [00:00<00:00, 3.02kB/s]
Metagenome plasmids (Acinetobacter): 100%|█████████████████████| 4.97k/4.97k [00:00<00:00, 6.62kB/s]
Metagenome plasmids (Enterococcus): 100%|██████████████████████| 4.63k/4.63k [00:00<00:00, 